## Core Changes

### 1. VRAM & Resource Optimization

- **`gpu_memory_utilization = 0.9`**  
  Reserves **10% of VRAM** for the operating system and fragmented allocations.  
  This improves **runtime stability**, especially when processing large batches or multiple concurrent requests.

---

### 2. Diversified Reasoning Strategies (System Prompts)

Adding a dataset contains 50 IMO QnA

The reasoning framework has been redesigned with **four specialized personas**:

- **Empiricist** — computation-driven exploration and numerical validation  
- **Pattern Combinator** — pattern discovery and combinatorial reasoning  
- **Formalist** — strict logical deduction and formal reasoning  
- **Backward Verifier** — solution-first reasoning with backward validation

With **`attempts = 8`**, the system distributes prompts **evenly (2 runs per persona)**.  
This rotation increases the **solution space coverage**, improving the effectiveness of the **majority voting strategy** across different reasoning styles.

---

### 3. Enforced Self-Verification (Assert Mechanism)

A **`CRITICAL` directive** is added to all **system prompts** and the **tool prompt**.

The model is required to:

- Implement **`assert` statements** in generated code
- Test **base cases**
- Validate **intermediate conditions**
- Verify **detected patterns or invariants** before computing the final answer

This mechanism enforces **self-checking behavior**, reducing silent reasoning errors.

---

### 4. Expanded Python Library Support (Preference Prompt)

The preferred Python toolset has been extended to improve coverage for **algorithmic and combinatorial strategies**.

New additions:

- **`scipy`** — numerical methods and scientific computation
- **`itertools`** — efficient combinatorial iteration and brute-force exploration

These are added alongside existing libraries:

- `math`
- `numpy`
- `sympy`

This expansion enables more **robust brute-force search, combinatorial enumeration, and symbolic reasoning workflows**.

# 🙏 Acknowledgements / Credits

This notebook is built upon the incredible work and open-source contributions of the Kaggle community. A massive thank you to the authors of the following notebooks for their foundational code, tool integrations, and brilliant execution strategies:

* **[44-50 let me over cook](https://www.kaggle.com/code/nihilisticneuralnet/44-50-let-me-over-cook)** by **nihilisticneuralnet** – For the robust generation loop, stable execution strategies, and inspiration for the "over-cook" approach.

* **[AIMO 3: GPT-OSS 120B with tools](https://www.kaggle.com/code/andreasbis/aimo-3-gpt-oss-120b-with-tools)** by **andreasbis** – For the brilliant foundational implementation of the GPT-OSS 120B model and the seamless Python tool integration via `openai_harmony`.

* **[Finetuned Model on Fork](https://www.kaggle.com/code/huikang/finetuned-model-on-fork)** by **huikang** – For the advanced entropy-based scoring approach, improved token management strategy (`buffer_tokens` and `search_tokens`), and the model weight preloading optimization that significantly improves initialization speed.

If you find this notebook helpful, please consider checking out and upvoting their original work!


# 1. **🧹 Environment Cleanup & Setup**

We begin by uninstalling heavy, unnecessary frameworks. Since this pipeline relies strictly on **vLLM** and **PyTorch** for LLM inference, removing the following libraries helps free up valuable disk space and prevents dependency conflicts in the Kaggle environment:

- ❌ `keras`
- ❌ `matplotlib`
- ❌ `scikit-learn`
- ❌ `tensorflow`

After cleaning the environment, we also import our **standard built-in Python libraries** that will be used throughout the pipeline.

⚙️ This step ensures a **lightweight, conflict-free runtime** optimized for efficient LLM inference.

In [ ]:
%pip uninstall -q --yes 'keras' 'matplotlib' 'scikit-learn' 'tensorflow'

import warnings
warnings.simplefilter('ignore')

import os, sys, subprocess, gc, re, math, time, queue, threading, contextlib

# 2. **📦 Offline Package Installation**
Because the Kaggle competition environment disables internet access, we cannot install packages directly from **PyPI**.

To solve this, we use a function that:

- 📂 Extracts a **pre-uploaded archive (`.tar.gz`)** containing compiled wheels  
- 📚 Includes dependencies such as:
  - `vllm`
  - `unsloth`
  - `openai_harmony`
- 🔍 Installs packages using the `--find-links` flag so `pip` searches **only inside the extracted directory**

⚙️ This allows all dependencies to be installed **completely offline**, ensuring the pipeline runs smoothly within Kaggle’s restricted environment.

In [ ]:
def set_env(archive, tmp):
    if not os.path.exists(tmp):
        os.makedirs(tmp, exist_ok=True)
        subprocess.run(['tar', '-xzf', archive, '-C', tmp], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--find-links', f'{tmp}/wheels', 
                    'unsloth', 'trl', 'vllm', 'openai_harmony'], check=True)

set_env('/kaggle/input/aimo-3-utils/wheels.tar.gz', '/kaggle/tmp/setup')

In [ ]:
subprocess.run(['ls', '/kaggle/tmp/setup/tiktoken_encodings'])

# 3. 🌐 **Environment Variables & Inference Tuning**

In this step, we configure the **system environment** to guarantee strict **offline operation** and stable inference behavior.

Key configurations include:

- 🚫 **Disable unused frameworks**  
  - Set `TRANSFORMERS_NO_TF=1` to prevent Hugging Face from loading **TensorFlow** or **Flax**, ensuring the pipeline relies only on **PyTorch**.

- 🎮 **Explicit CUDA configuration**  
  - Define the CUDA device so GPU resources are used correctly during inference.

- 🧠 **Local tokenizer resources**  
  - Point the tokenizer to our **locally extracted `tiktoken` encodings** to avoid any attempt to download files from the internet.

⚙️ These settings ensure the pipeline runs **fully offline**, avoids unnecessary framework initialization, and keeps **LLM inference stable and efficient**.

In [ ]:
for k, v in [('TRANSFORMERS_NO_TF', '1'), ('TRANSFORMERS_NO_FLAX', '1'), ('CUDA_VISIBLE_DEVICES', '0'),
             ('TOKENIZERS_PARALLELISM', 'false'), ('TRITON_PTXAS_PATH', '/usr/local/cuda/bin/ptxas'),
             ('TIKTOKEN_ENCODINGS_BASE', '/kaggle/tmp/setup/tiktoken_encodings')]:
    os.environ[k] = v

# 4. 📚 **Core Library Imports**

In this step, we import the specialized modules required for our **inference pipeline**.

Key components include:

- ⚡ **Parallel execution**  
  - `ThreadPoolExecutor` enables **asynchronous parallel tasks**, allowing multiple operations to run concurrently and improving throughput.

- 🧩 **Stateful sandbox management**  
  - `jupyter_client` is used to manage **stateful execution environments**, which helps maintain controlled execution contexts during runtime.

- 📊 **Data processing**  
  - `polars` and `pandas` handle **efficient data loading, transformation, and analysis** throughout the pipeline.

- 📨 **Structured API communication**  
  - `openai_harmony` ensures **strict message formatting** when interacting with the model API, keeping request and response structures consistent.

⚙️ These imports form the **core infrastructure layer** that supports parallelism, data handling, environment control, and reliable model communication.

In [ ]:
from typing import Optional
from jupyter_client import KernelManager
from collections import Counter, defaultdict
from concurrent.futures import as_completed, ThreadPoolExecutor
import pandas as pd, polars as pl
from openai import OpenAI
from openai_harmony import (HarmonyEncodingName, load_harmony_encoding, SystemContent, ReasoningEffort, 
                             ToolNamespaceConfig, Author, Message, Role, TextContent, Conversation)
from transformers import set_seed
import kaggle_evaluation.aimo_3_inference_server

# 5. ⚙️ **Global Configuration & Modified System Prompt**

The `CFG` class stores the **global parameters** that control our entire pipeline.

🔑 **Key Implementation: Code-Verifier Strategy**

The `system_prompt` is intentionally designed to instruct the model that it **must produce a final Python script** to compute its answer.

This design activates our **🧪 Code Verifier Filter**, which works as follows:

1. 🧠 The LLM generates reasoning **and a final executable Python script**.
2. 🏃 The script is executed inside a **secure sandbox environment**.
3. 📐 The computation result is used to **mathematically verify the model’s logic**.
4. 🗳️ The verified result then participates in **majority voting** across multiple model outputs.

✅ By forcing the model to produce **executable code**, we transform free-form reasoning into **verifiable computation**, greatly improving reliability and correctness.

In [ ]:
class CFG:
    system_prompts = [
            ('You are an expert in computational mathematics. '
             'Solve the problem by writing Python scripts to compute base cases, simulate scenarios, or perform bounded searches. '
             'CRITICAL: Always write assert statements for small test cases before scaling up. '
             'The final answer must be a non-negative integer between 0 and 99999 inside \\boxed{}.'),
            
            ('You are a master of heuristic problem-solving. '
             'First, manually compute the results for small instances (e.g., n=1, 2, 3) to find invariants or patterns. '
             'Then, write Python code to verify your hypothesis. '
             'CRITICAL: Use assert to check your pattern against base cases. '
             'The final answer must be a non-negative integer between 0 and 99999 inside \\boxed{}.'),
             
            ('You are a rigorous mathematical formalist.'
             'Solve the problem using rigorous algebraic formalization. Follow these exact steps:'
             'Write a Python script heavily utilizing the sympy library to symbolically solve these equations '
             'CRITICAL: Write assert statements to verify that the roots or solutions found satisfy the boundaries of the original problem before formatting the final output.'
             'The final answer must be a non-negative integer between 0 and 99999 inside \\boxed{}.'),
             
            ('You are an expert at working backwards. '
             'Make educated deductions for the solution boundaries, then write Python code to construct the mathematical conditions and verify which values hold true. '
             'CRITICAL: assert that your final answer satisfies ALL original problem conditions. '
             'The final answer must be a non-negative integer between 0 and 99999 inside \\boxed{}.')
        ]
        
    tool_prompt = ('Use this tool to execute Python code in a stateful Jupyter notebook environment. Directives: '
                       'Keep code modular and computationally efficient. '
                       'You MUST write assert statements to verify base cases (e.g., n=1, n=2) before running large loops or searches. '
                       'Always use print() to output the final integer result so the environment can capture it. '
                       'If an execution errors out, analyze the traceback and rewrite the code."'
                      )
    preference_prompt = 'You have access to `math`, `numpy`, `scipy`, `itertools`, and `sympy` to solve the problem.'
    served_model_name, model_path = 'gpt-oss', '/kaggle/input/models/huikang/gpt-oss-120b-aimo3/transformers/160a/9'
    kv_cache_dtype, dtype = 'fp8_e4m3', 'auto'
    high_problem_timeout, base_problem_timeout = 900, 270
    notebook_limit, server_timeout = 17400, 180
    session_timeout, jupyter_timeout, sandbox_timeout = 960, 6, 3
    stream_interval, context_tokens, buffer_tokens, search_tokens = 200, 65536, 512, 32 
    top_logprobs, batch_size, early_stop, attempts, workers, turns = 5, 128, 3, 8, 16, 128 
    gpu_memory_utilization, temperature, min_p, seed = 0.9, 1.0, 0.02, 42

In [ ]:
set_seed(CFG.seed)

# 6. 🧩 **Prompt Templating & API Formatting**

The `AIMO3Template` class acts as the bridge between our **string-based configurations** and the **API server interface**.

🔑 **Key Responsibilities**

- 📨 **Structured messaging**  
  - Uses `openai_harmony` to convert raw prompt text into **strict `Message` objects**, ensuring the request follows the expected API format.

- 🧠 **High-effort reasoning configuration**  
  - Applies `ReasoningEffort.HIGH`, which instructs the reasoning model to allocate **maximum compute tokens** to its internal reasoning process before producing the final output.

- ⚙️ **Prompt standardization**  
  - Ensures prompts are **consistently formatted**, reducing variability and improving stability across inference calls.

✅ This layer guarantees that prompts are **cleanly structured, API-compatible, and optimized for deeper reasoning performance**.

In [ ]:
class AIMO3Template:
    def get_system_content(self, prompt, tool_cfg):
        return SystemContent.new().with_model_identity(prompt).with_reasoning_effort(
            reasoning_effort=ReasoningEffort.HIGH).with_tools(tool_cfg)
    
    def apply_chat_template(self, sys_prompt, usr_prompt, tool_cfg):
        return [Message.from_role_and_content(Role.
                                              SYSTEM, self.get_system_content(sys_prompt, tool_cfg)),
                Message.from_role_and_content(Role.USER, usr_prompt)]

# 7. 🧪 **Python Execution Sandbox (`AIMO3Sandbox`)**

This class creates the **isolated Python execution environment** used to validate the LLM’s generated code.

🔑 **Core Mechanism**

- 🧠 **Stateful Python kernels**  
  - Uses `jupyter_client.KernelManager` to launch **persistent background Python processes**, allowing code to run in a controlled, reusable environment.

- 📚 **Preloaded math libraries**  
  - Automatically imports heavy computational libraries during initialization:
    - `sympy`
    - `numpy`
    - `mpmath`  
  - This saves **generation tokens**, since the LLM does not need to repeatedly write import statements.

- ⏱️ **Execution safety controls**  
  - Enforces **strict execution timeouts** to prevent infinite loops or excessively long computations produced by the model.

⚙️ The sandbox ensures that **LLM-generated Python code can be executed safely, efficiently, and reproducibly** while verifying mathematical reasoning.

In [ ]:
class AIMO3Sandbox:
    _port_lock, _next_port = threading.Lock(), 50000
    
    @classmethod
    def _get_next_ports(cls, count=5):
        with cls._port_lock:
            ports = list(range(cls._next_port, cls._next_port + count))
            cls._next_port += count
            return ports
    
    def __init__(self, timeout):
        self._default_timeout, self._owns_kernel, self._client, self._km = timeout, False, None, None
        ports = self._get_next_ports(5)
        env = os.environ.copy()
        env.update({'PYDEVD_DISABLE_FILE_VALIDATION': '1', 'PYDEVD_WARN_EVALUATION_TIMEOUT': '0',
                   'JUPYTER_PLATFORM_DIRS': '1', 'PYTHONWARNINGS': 'ignore', 'MPLBACKEND': 'Agg'})
        self._km = KernelManager()
        self._km.shell_port, self._km.iopub_port, self._km.stdin_port, self._km.hb_port, self._km.control_port = ports
        self._km.start_kernel(env=env, extra_arguments=['--Application.log_level=CRITICAL'])
        self._client = self._km.blocking_client()
        self._client.start_channels()
        self._client.wait_for_ready(timeout=self._default_timeout)
        self._owns_kernel = True
        self.execute('import math, numpy, sympy, mpmath, itertools, collections\nmpmath.mp.dps = 64\n')
    
    def _format_error(self, tb):
        return ''.join(re.sub(r'\x1b\[[0-9;]*m', '', f) for f in tb 
                      if 'File "' not in f or 'ipython-input' in f)
    
    def execute(self, code, timeout=None):
        effective_timeout = timeout or self._default_timeout
        msg_id = self._client.execute(code, store_history=True, allow_stdin=False, stop_on_error=False)
        stdout, stderr, start = [], [], time.time()
        while True:
            if time.time() - start > effective_timeout:
                self._km.interrupt_kernel()
                return f'[ERROR] Execution timed out after {effective_timeout} seconds'
            try:
                msg = self._client.get_iopub_msg(timeout=1.0)
            except queue.Empty:
                continue
            if msg.get('parent_header', {}).get('msg_id') != msg_id: continue
            mt, c = msg.get('msg_type'), msg.get('content', {})
            if mt == 'stream':
                (stdout if c.get('name') == 'stdout' else stderr).append(c.get('text', ''))
            elif mt == 'error':
                stderr.append(self._format_error(c.get('traceback', [])))
            elif mt in {'execute_result', 'display_data'}:
                if txt := c.get('data', {}).get('text/plain'):
                    stdout.append(txt if txt.endswith('\n') else f'{txt}\n')
            elif mt == 'status' and c.get('execution_state') == 'idle':
                break
        out, err = ''.join(stdout), ''.join(stderr)
        return f'{out.rstrip()}\n{err}' if err and out else (err or out or '[WARN] No output. Use print() to see results.')
    
    def close(self):
        with contextlib.suppress(Exception):
            if self._client: self._client.stop_channels()
        if self._owns_kernel and self._km:
            with contextlib.suppress(Exception): self._km.shutdown_kernel(now=True)
            with contextlib.suppress(Exception): self._km.cleanup_resources()
    
    def reset(self):
        self.execute('%reset -f\nimport math, numpy, sympy, mpmath, itertools, collections\nmpmath.mp.dps = 64\n')
    
    def __del__(self):
        self.close()

# 8. 🛠️ **LLM Tool Integration (`AIMO3Tool`)**

This class registers the **`python` execution environment** as a **callable tool** that the LLM can use during reasoning.

🔑 **Core Responsibilities**

- 🧠 **Tool exposure to the model**  
  - The `python` sandbox becomes an **available tool**, allowing the LLM to execute generated code during the reasoning process.

- 🔄 **Automatic output correction**  
  - The `_ensure_last_print` method acts as an **auto-correction wrapper** for generated scripts.

⚙️ **How the correction works**

1. The LLM generates executable Python code.
2. Sometimes the model **forgets to call `print()`** on the final result.
3. `_ensure_last_print` intercepts the script.
4. If needed, it **wraps the last expression in a `print()` statement**.
5. The result is then **captured and returned to the model**.

✅ This mechanism guarantees that **computed outputs are always visible**, ensuring reliable feedback between the LLM and the execution sandbox.

In [ ]:
class AIMO3Tool:
    def __init__(self, timeout, prompt, sandbox=None):
        self._local_jupyter_timeout, self._tool_prompt, self._jupyter_session = timeout, prompt, sandbox
        self._owns_session, self._execution_lock, self._init_lock = sandbox is None, threading.Lock(), threading.Lock()
    
    def _ensure_session(self):
        if self._jupyter_session is None:
            with self._init_lock:
                if self._jupyter_session is None:
                    self._jupyter_session = AIMO3Sandbox(timeout=self._local_jupyter_timeout)
    
    def _ensure_last_print(self, code):
        lines = code.strip().split('\n')
        if not lines: return code
        last = lines[-1].strip()
        if any(x in last for x in ['print', 'import']) or not last or last.startswith('#'): return code
        lines[-1] = 'print(' + last + ')'
        return '\n'.join(lines)
    
    @property
    def instruction(self): return self._tool_prompt
    
    @property
    def tool_config(self): return ToolNamespaceConfig(name='python', description=self.instruction, tools=[])
    
    def _make_response(self, output, channel=None):
        msg = Message(author=Author(role=Role.TOOL, name='python'), 
                     content=[TextContent(text=output)]).with_recipient('assistant')
        return msg.with_channel(channel) if channel else msg
    
    def process_sync_plus(self, message):
        self._ensure_session()
        final_script = self._ensure_last_print(message.content[0].text)
        with self._execution_lock:
            try:
                output = self._jupyter_session.execute(final_script)
            except TimeoutError as exc:
                output = f'[ERROR] {exc}'
        return [self._make_response(output, channel=message.channel)]

# 9. 🧠 **Master Orchestrator: `AIMO3Solver`**  
*(Part 1 — Setup & Server Initialization)*

The `AIMO3Solver` class acts as the **central engine** of the entire pipeline.

🔑 **Core Responsibilities**

- 🚀 **Local vLLM server startup**  
  - Launches a **local `vLLM` inference server** inside a background subprocess, enabling fast local model inference.

- 💾 **Model weight caching**  
  - Preloads model weights into the **OS Page Cache**, reducing disk I/O latency and speeding up repeated inference calls.

- 🧪 **Sandbox pool initialization**  
  - Creates a **pool of Jupyter execution sandboxes** (threads) so multiple code verification tasks can run in parallel.

- 📊 **Dynamic entropy monitoring**  
  - Continuously tracks **token generation entropy** during decoding.

⚙️ **Dynamic Entropy Abort Strategy**

If entropy becomes excessively high (indicating unstable or low-confidence generation), the system can **terminate the generation early**, preventing wasted computation on low-quality reasoning paths.

✅ This orchestration layer manages **model serving, execution environments, and inference monitoring**, ensuring the system runs efficiently at scale.

In [ ]:
class AIMO3Solver:
    def __init__(self, cfg, port=8000):
        self.cfg, self.port = cfg, port
        self.base_url, self.api_key = f'http://0.0.0.0:{port}/v1', 'sk-local'
        self.template, self.encoding = AIMO3Template(), load_harmony_encoding(HarmonyEncodingName.HARMONY_GPT_OSS)
        self.stop_token_ids = self.encoding.stop_tokens_for_assistant_actions()
        self._preload_model_weights()
        self.server_process = self._start_server()
        self.client = OpenAI(base_url=self.base_url, api_key=self.api_key, timeout=self.cfg.session_timeout)
        self._wait_for_server()
        self._initialize_kernels()
        self.notebook_start_time, self.problems_remaining = time.time(), 50
    
    def _preload_model_weights(self):
        print(f'Loading model weights from {self.cfg.model_path} into OS Page Cache...')
        start, files, total = time.time(), [], 0
        for root, _, fnames in os.walk(self.cfg.model_path):
            for fn in fnames:
                fp = os.path.join(root, fn)
                if os.path.isfile(fp):
                    files.append(fp)
                    total += os.path.getsize(fp)
        with ThreadPoolExecutor(max_workers=self.cfg.workers) as ex:
            list(ex.map(lambda p: open(p, 'rb').read(), files))
        print(f'Processed {len(files)} files ({total/1e9:.2f} GB) in {time.time()-start:.2f} seconds.\n')
    
    def _start_server(self):
        cmd = [sys.executable, '-m', 'vllm.entrypoints.openai.api_server', '--seed', str(self.cfg.seed),
               '--model', self.cfg.model_path, '--served-model-name', self.cfg.served_model_name,
               '--tensor-parallel-size', '1', '--max-num-seqs', str(self.cfg.batch_size),
               '--gpu-memory-utilization', str(self.cfg.gpu_memory_utilization), '--host', '0.0.0.0',
               '--port', str(self.port), '--dtype', self.cfg.dtype, '--kv-cache-dtype', self.cfg.kv_cache_dtype,
               '--max-model-len', str(self.cfg.context_tokens), '--stream-interval', str(self.cfg.stream_interval),
               '--async-scheduling', '--disable-log-stats', '--enable-prefix-caching']
        self.log_file = open('vllm_server.log', 'w')
        return subprocess.Popen(cmd, stdout=self.log_file, stderr=subprocess.STDOUT, start_new_session=True)
    
    def _wait_for_server(self):
        print('Waiting for vLLM server...')
        start = time.time()
        for _ in range(self.cfg.server_timeout):
            if (rc := self.server_process.poll()) is not None:
                self.log_file.flush()
                raise RuntimeError(f'Server died with code {rc}. Full logs:\n{open("vllm_server.log").read()}\n')
            try:
                self.client.models.list()
                print(f'Server is ready (took {time.time()-start:.2f} seconds).\n')
                return
            except Exception:
                time.sleep(1)
        raise RuntimeError('Server failed to start (timeout).\n')
    
    def _initialize_kernels(self):
        print(f'Initializing {self.cfg.workers} persistent Jupyter kernels...')
        start = time.time()
        self.sandbox_pool = queue.Queue()
        with ThreadPoolExecutor(max_workers=self.cfg.workers) as ex:
            for future in as_completed([ex.submit(lambda: AIMO3Sandbox(timeout=self.cfg.jupyter_timeout)) 
                                       for _ in range(self.cfg.workers)]):
                self.sandbox_pool.put(future.result())
        print(f'Kernels initialized in {time.time()-start:.2f} seconds.\n')
    
    def _scan_for_answer(self, text):
        for pattern in [r'\\boxed\s*\{\s*([0-9,]+)\s*\}', r'final\s+answer\s+is\s*([0-9,]+)']:
            if matches := re.findall(pattern, text, re.IGNORECASE):
                try:
                    val = int(matches[-1].replace(',', ''))
                    if 0 <= val <= 99999: return val
                except ValueError: pass
        return None
    
    def _compute_mean_entropy(self, logprobs):
        """
        Compute weighted entropy metric optimized for mathematical reasoning quality.
        Lower entropy indicates more confident, focused reasoning.
        
        Key improvements over simple mean:
        1. Position weighting - recent tokens (near final answer) matter more
        2. Consistency penalty - variance in confidence indicates uncertain reasoning
        3. Sustained uncertainty penalty - long stretches of high entropy are bad
        4. Confidence streak reward - consistent low entropy indicates strong reasoning
        5. Calibrated for mathematical problem-solving patterns
        """
        if not logprobs: 
            return float('inf')
        
        entropies = []
        for top_lp in logprobs:
            if isinstance(top_lp, dict) and top_lp:
                # Shannon entropy in bits: H = -Σ p(x) * log2(p(x))
                ent = sum(-math.exp(lp)*math.log2(math.exp(lp)) for lp in top_lp.values() if math.exp(lp) > 0)
                entropies.append(ent)
        
        if not entropies: 
            return float('inf')
        
        n = len(entropies)
        
        # Component 1: Base mean entropy (baseline uncertainty)
        mean_ent = sum(entropies) / n
        
        # Component 2: Variance penalty (penalize inconsistent confidence)
        # Math problems should show steady confidence, not wild swings
        variance = sum((e - mean_ent)**2 for e in entropies) / n
        std_dev = math.sqrt(variance)
        
        # Component 3: Position-weighted entropy (exponential decay)
        # Tokens closer to the final answer are more important
        # decay_factor < 1 means recent tokens get exponentially more weight
        decay_factor = 0.995
        weighted_sum = sum(e * (decay_factor ** (n - i - 1)) for i, e in enumerate(entropies))
        weighted_count = sum(decay_factor ** (n - i - 1) for i in range(n))
        position_weighted_ent = weighted_sum / weighted_count if weighted_count > 0 else mean_ent
        
        # Component 4: Sustained high entropy penalty
        # Long periods of uncertainty suggest the model is lost/guessing
        high_ent_threshold = 2.0  # bits (adjust based on your model's typical range)
        high_ent_ratio = sum(1 for e in entropies if e > high_ent_threshold) / n
        
        # Component 5: Low entropy streak bonus
        # Reward long sequences of confident predictions (good reasoning chains)
        low_ent_threshold = 0.5  # bits
        max_streak = 0
        current_streak = 0
        for e in entropies:
            if e < low_ent_threshold:
                current_streak += 1
                max_streak = max(max_streak, current_streak)
            else:
                current_streak = 0
        
        # Normalize streak by sequence length and convert to bonus (negative reduces final entropy)
        streak_bonus = -0.1 * (max_streak / n)
        
        # Final weighted combination
        # Weights tuned for math reasoning (position-weighted is most important)
        final_entropy = (
            0.3 * mean_ent +                    # Base uncertainty level
            0.4 * position_weighted_ent +       # Recent token confidence (MOST IMPORTANT)
            0.2 * std_dev +                     # Consistency of confidence
            0.3 * high_ent_ratio * 3.0 +        # Heavy penalty for sustained uncertainty
            streak_bonus                         # Bonus for confident reasoning chains
        )
        
        return final_entropy
    
    def _process_attempt(self, problem, sys_prompt, idx, stop_evt, deadline):
        if stop_evt.is_set() or time.time() > deadline:
            return {'Attempt': idx+1, 'Answer': None, 'Python Calls': 0, 'Python Errors': 0, 
                   'Response Length': 0, 'Entropy': float('inf')}
        local_tool, sandbox, py_calls, py_errs, total_toks, ans, logprobs = None, None, 0, 0, 0, None, []
        seed = int(math.pow(self.cfg.seed + idx, 2))
        try:
            sandbox = self.sandbox_pool.get(timeout=self.cfg.sandbox_timeout)
            local_tool = AIMO3Tool(self.cfg.jupyter_timeout, self.cfg.tool_prompt, sandbox)
            conv = Conversation.from_messages(self.template.apply_chat_template(
                sys_prompt, problem, local_tool.tool_config))
            for _ in range(self.cfg.turns):
                if stop_evt.is_set() or time.time() > deadline: break
                prompt_ids = self.encoding.render_conversation_for_completion(conv, Role.ASSISTANT)
                if (max_toks := self.cfg.context_tokens - len(prompt_ids)) < self.cfg.buffer_tokens: break
                stream = self.client.completions.create(model=self.cfg.served_model_name, 
                    temperature=self.cfg.temperature, logprobs=self.cfg.top_logprobs, max_tokens=max_toks,
                    prompt=prompt_ids, seed=seed, stream=True, extra_body={
                        'min_p': self.cfg.min_p, 'stop_token_ids': self.stop_token_ids, 'return_token_ids': True})
                try:
                    tok_buf, txt_chunks = [], []
                    for chunk in stream:
                        if stop_evt.is_set() or time.time() > deadline: break
                        if new_toks := chunk.choices[0].token_ids:
                            tok_buf.extend(new_toks)
                            total_toks += len(new_toks)
                            txt_chunks.append(chunk.choices[0].text)
                            if (clp := chunk.choices[0].logprobs) and clp.top_logprobs:
                                logprobs.extend(clp.top_logprobs)
                        if '}' in chunk.choices[0].text and (ans := self._scan_for_answer(
                            ''.join(txt_chunks[-self.cfg.search_tokens:]))):
                            break
                finally:
                    stream.close()
                if ans or not tok_buf: break
                new_msgs = self.encoding.parse_messages_from_completion_tokens(tok_buf, Role.ASSISTANT)
                conv.messages.extend(new_msgs)
                last = new_msgs[-1]
                if last.channel == 'final':
                    ans = self._scan_for_answer(last.content[0].text)
                    break
                # Log the full reasoning trace, generated code, and encountered errors to aimo_reasoning_log.jsonl after each attempt for debugging and traceability.
                if last.recipient == 'python':
                    py_calls += 1
                    resp = local_tool.process_sync_plus(last)
                    if any(x in (txt := resp[0].content[0].text) for x in ['[ERROR]', 'Traceback', 'Error:']):
                        py_errs += 1
                    conv.messages.extend(resp)
            
            import json
            log_data = {
                'attempt': idx + 1,
                'answer': ans,
                'python_calls': py_calls,
                'python_errors': py_errs,
                'entropy': self._compute_mean_entropy(logprobs),
                'history': [{'role': str(m.author.role), 'content': getattr(m.content[0], 'text', str(m.content[0])) if getattr(m, 'content', None) else ''} for m in conv.messages]
            }
            with open('aimo_reasoning_log.jsonl', 'a') as f:
                f.write(json.dumps(log_data) + '\n')
                
        except Exception as e:
            print(f"[DEBUG] Attempt {idx+1} failed: {repr(e)}")
            py_errs += 1
        finally:
            if sandbox:
                sandbox.reset()
                self.sandbox_pool.put(sandbox)
        return {'Attempt': idx+1, 'Response Length': total_toks, 'Python Calls': py_calls, 
               'Python Errors': py_errs, 'Entropy': self._compute_mean_entropy(logprobs), 'Answer': ans}
    
    def _select_answer(self, results):
        ans_weights, ans_votes = defaultdict(float), defaultdict(int)
        for r in results:
            if (a := r['Answer']) is not None:
                ans_weights[a] += 1.0/max(r['Entropy'], 1e-9)
                ans_votes[a] += 1
        scored = sorted([{'answer': a, 'votes': ans_votes[a], 'score': w} 
                        for a, w in ans_weights.items()], key=lambda x: x['score'], reverse=True)
        display(pd.DataFrame([(s['answer'], s['votes'], s['score']) for s in scored], 
                            columns=['Answer', 'Votes', 'Score']).round({'Score': 3}))
        final = scored[0]['answer'] if scored else 0
        print(f'\nFinal Answer: {final}\n')
        return final
    
    def solve_problem(self, problem):
        print(f'\nProblem: {problem}\n')
        user_input = f'{problem} {self.cfg.preference_prompt}'
        time_left = self.cfg.notebook_limit - (time.time() - self.notebook_start_time)
        budget = max(self.cfg.base_problem_timeout, 
                    min(time_left - max(0, self.problems_remaining-1)*self.cfg.base_problem_timeout, 
                        self.cfg.high_problem_timeout))
        deadline = time.time() + budget
        print(f'Budget: {budget:.2f} seconds | Deadline: {deadline:.2f}\n')
        results, valid, stop_evt = [], [], threading.Event()
        with ThreadPoolExecutor(max_workers=self.cfg.workers) as ex:
            futures = [ex.submit(self._process_attempt, user_input, self.cfg.system_prompts[i % len(self.cfg.system_prompts)], i, stop_evt, deadline)
                      for i in range(self.cfg.attempts)]
            for future in as_completed(futures):
                try:
                    if (r := future.result())['Answer'] is not None:
                        valid.append(r['Answer'])
                    results.append(r)
                    if (cnts := Counter(valid).most_common(1)) and cnts[0][1] >= self.cfg.early_stop:
                        stop_evt.set()
                        for f in futures: f.cancel()
                        break
                except Exception as exc:
                    print(f'Future failed: {exc}')
        self.problems_remaining = max(0, self.problems_remaining - 1)
        if results:
            df = pd.DataFrame(results)
            df['Entropy'] = df['Entropy'].round(3)
            df['Answer'] = df['Answer'].astype('Int64')
            display(df)
        return self._select_answer(results) if valid else 0
    
    def __del__(self):
        if hasattr(self, 'server_process'):
            self.server_process.terminate()
            self.server_process.wait()
        if hasattr(self, 'log_file'): self.log_file.close()
        if hasattr(self, 'sandbox_pool'):
            while not self.sandbox_pool.empty():
                with contextlib.suppress(Exception): self.sandbox_pool.get_nowait().close()

# 10. 🚀 **Initialization & Kaggle Evaluation Server**

With the core classes defined, we now **initialize the pipeline** and prepare it for Kaggle’s evaluation environment.

🔑 **Main Steps**

- ⚙️ **Solver initialization**  
  - Instantiate the `AIMO3Solver`, which activates the inference engine, sandbox pool, and model server.

- 📂 **Dataset duplication for simulation**  
  - Duplicate the **reference dataset** to simulate **commit-time evaluation runs**, ensuring the pipeline behaves the same way during real Kaggle submissions.

- 🔮 **Prediction wrapper function**  
  - Define the standard `predict()` function that receives **incoming test variables**, processes them through the solver, and returns the final prediction.

- 🌐 **Inference server startup**  
  - Launch the `AIMO3InferenceServer`, which dynamically serves predictions during Kaggle’s evaluation process.

✅ This stage connects the **solver, dataset simulation, and prediction interface**, enabling the pipeline to operate smoothly inside Kaggle’s **automated evaluation system**.

In [ ]:
solver = AIMO3Solver(CFG)

In [ ]:
import os

import kaggle_evaluation.aimo_3_inference_server
import pandas as pd
import polars as pl

def is_on_kaggle() -> bool:
    return bool(os.getenv("KAGGLE_KERNEL_RUN_TYPE"))

if is_on_kaggle():
    df = pd.read_csv(
        "/kaggle/input/ai-mathematical-olympiad-progress-prize-3/reference.csv"
    ).drop("answer", axis=1)
else:
    df = pd.read_csv("aimo3.csv").drop("answer", axis=1)

dfs = []
replication_count_for_commit_runs = 5
for replication_idx in range(replication_count_for_commit_runs):
    df_copy = df.copy()
    df_copy["id"] = df_copy["id"] + f"_{replication_idx}"
    dfs.append(df_copy)
pd.concat(dfs, ignore_index=True).to_csv("reference.csv", index=False)

In [ ]:
def predict(id_: pl.DataFrame, question: pl.DataFrame, answer: Optional[pl.DataFrame] = None) -> pl.DataFrame:
    
    id_value = id_.item(0)
    question_text = question.item(0)
    
    gc.disable()
    
    final_answer = solver.solve_problem(question_text)
    
    gc.enable()
    gc.collect()
    
    return pl.DataFrame({'id': id_value, 'answer': final_answer})


In [ ]:
%%time
inference_server = kaggle_evaluation.aimo_3_inference_server.AIMO3InferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
    
else:
    inference_server.run_local_gateway(
        ('/kaggle/input/ai-mathematical-olympiad-progress-prize-3/test.csv',)
    )

# 11. 🏁 **Testing Samples (Ultimate Benchmark Suite)**

This section provides a **robust benchmark suite of 20 challenging math problems**, each paired with its **exact ground-truth solution**.

🔑 **Benchmark Workflow**

- 🧠 **Problem iteration**  
  - The system sequentially processes each problem, triggering the full inference pipeline for evaluation.

- 🧹 **Memory management**  
  - Explicit **garbage collection** is performed between runs to prevent **out-of-memory (OOM) errors**, which is especially important when working with large models.

- 📊 **Performance evaluation**  
  - Results are aggregated into a **styled DataFrame** for clear visualization.

📈 **Reported Metrics**

- 🎯 **Overall accuracy** — compares predicted results against the ground-truth values  
- ⏱️ **Execution speed** — measures how quickly the pipeline solves each problem

✅ This benchmark acts as an **ultimate validation layer**, stress-testing the pipeline’s **correctness, stability, and runtime efficiency**.

In [ ]:
'''import time
import gc
import pandas as pd
from IPython.display import display

# ==============================================================================
# ULTIMATE AIMO BENCHMARK SUITE (20 PROBLEMS) WITH GROUND TRUTH VALIDATION
# ==============================================================================

test_suite = [
    {
        "problem": "Find the sum of all positive integers $n$ such that $n^2 + 2025$ is a perfect square.",
        "truth": 1768
    },
    {
        "problem": "Let $a, b, c,$ and $d$ be real numbers satisfying $a+b+c+d=4$, $a^2+b^2+c^2+d^2=10$, $a^3+b^3+c^3+d^3=22$, and $a^4+b^4+c^4+d^4=46$. Calculate the exact integer value of $a^6+b^6+c^6+d^6$.",
        "truth": 184
    },
    {
        "problem": "Let $S$ be the number of ways to distribute 20 identical candies to 5 distinct children such that no child receives more than 6 candies. Find the remainder when $S$ is divided by $100000$.",
        "truth": 826
    },
    {
        "problem": "In triangle $ABC$, the side lengths are $AB=13$, $BC=14$, and $CA=15$. Let $I$ be the incenter and $O$ be the circumcenter of the triangle. Calculate the exact integer value of $1024 \\times IO^2$.",
        "truth": 1040 # R = 65/8, r = 4. IO^2 = R(R-2r) = 65/8 * (65/8 - 8) = 65/8 * 1/8 = 65/64. 1024 * 65/64 = 16 * 65 = 1040.
    },
    {
        "problem": "How many sequences of 0s and 1s of length 15 are there such that they do not contain three consecutive 1s anywhere in the sequence?",
        "truth": 10609 # Tribonacci sequence offset T(18) = 10609
    },
    {
        "problem": "Find the number of ordered triples $(a,b,c)$ of positive integers such that $\\operatorname{lcm}(a,b) = 1000$, $\\operatorname{lcm}(b,c) = 2000$, and $\\operatorname{lcm}(c,a) = 2000$.",
        "truth": 70 # Needs prime factorization logic
    },
    {
        "problem": "Find the remainder when the number of distinct spanning trees of the complete bipartite graph $K_{5,5}$ is divided by 100000.",
        "truth": 90625 # Cayley's formula for bipartite: m^(n-1) * n^(m-1) = 5^4 * 5^4 = 5^8 = 390625. Mod 100000 = 90625.
    },
    {
        "problem": "Let $P(x)$ be a monic polynomial of degree 10 such that $P(k) = k^2$ for $k=1, 2, \\dots, 10$. Find the remainder when $P(11)$ is divided by $100000$.",
        "truth": 28921 # P(x) = (x-1)(x-2)...(x-10) + x^2. P(11) = 10! + 121 = 3628800 + 121 = 3628921. Mod 100000 = 28921.
    },
    {
        "problem": "Find the number of non-congruent triangles with integer side lengths whose perimeter is exactly 1000.",
        "truth": 20833 # Formula for perimeter p=1000: round(p^2 / 48) = 1000000/48 = 20833.33 -> 20833
    },
    {
        "problem": "Calculate the exact integer value of $100 \\times \\sum_{k=1}^{89} \\cos^2(k^\\circ)$.",
        "truth": 4450 # Sum = 44.5. 100 * 44.5 = 4450.
    },
    {
        "problem": "What is the remainder when $2025^{2025^{2025}}$ is divided by $10000$?",
        "truth": 8125 # Tricky modular arithmetic / Euler's Totient
    },
    {
        "problem": "A bug starts at the origin $(0,0)$ on a Cartesian plane. Each second, it moves 1 unit up, down, left, or right with equal probability. Let $P$ be the probability that it is back at the origin after exactly 10 seconds. Calculate the remainder when $4^{10} \\times P$ is divided by $99999$.",
        "truth": 63504 # (10 C 5) * (10 C 5) = 252 * 252 = 63504
    },
    {
        "problem": "A sequence satisfies $a_0 = 1, a_1 = 3$, and $a_n = 4a_{n-1} - a_{n-2}$ for $n \\ge 2$. Find the remainder when $a_{15}$ is divided by $100000$.",
        "truth": 26953 # Matrix exponentiation calculation.
    },
    {
        "problem": "A regular tetrahedron has an edge length of 12. A sphere is tangent to all six edges of the tetrahedron. What is the square of the radius of this sphere?",
        "truth": 18 # Distance from center to edge. R_edge = a * sqrt(2) / 4. R^2 = a^2 * 2 / 16 = a^2 / 8. 144 / 8 = 18.
    },
    {
        "problem": "How many ordered pairs of positive integers $(x,y)$ satisfy the equation $x^2 - y^2 = 100000$?",
        "truth": 10 # Divisor counting logic for (x-y)(x+y) = 100000. Both must be even.
    },
    {
        "problem": "How many permutations of the set $\\{1, 2, \\dots, 9\\}$ have exactly two fixed points?",
        "truth": 66744 # (9 C 2) * Derangements(7) = 36 * 1854 = 66744.
    },
    {
        "problem": "Let $z_1, z_2, z_3$ be complex numbers such that $|z_1| = |z_2| = |z_3| = 1$ and $z_1 + z_2 + z_3 = 0$. Evaluate the exact integer value of $1000 \\times |z_1^3 + z_2^3 + z_3^3|$.",
        "truth": 3000 # They form an equilateral triangle. z1^3+z2^3+z3^3 - 3z1z2z3 = (z1+z2+z3)(...). So sum of cubes = 3z1z2z3. |3z1z2z3| = 3. 1000*3 = 3000.
    },
    {
        "problem": "Find the last 5 digits (represented as an integer) of the sum of the 5th powers of the first 100 positive integers: $\\sum_{i=1}^{100} i^5$.",
        "truth": 33300 # Faulhaber formula or modulo arithmetic loop.
    },
    {
        "problem": "Find the number of non-negative integer solutions to the equation $x+y+z+w = 100$ subject to the constraints $x \\le 20, y \\le 30, z \\le 40$, and $w \\le 50$.",
        "truth": 601 # Inclusion-Exclusion Principle.
    },
    {
        "problem": "An urn contains 50 red balls and 50 blue balls. Balls are drawn one by one without replacement. Let $E$ be the expected number of times a drawn ball has a different color from the immediately preceding drawn ball. Calculate the exact integer value of $1000 \\times E$.",
        "truth": 50505 # Linearity of expectation. Probability of change at step i is (50/100 * 50/99) * 2 = 50/99. E = 99 * (50/99) = 50. Wait, exact E = 50.505050... -> 1000 * 50.5050 = 50505
    }
]

# Adjust problem 16 ground truth to precise value
test_suite[15]["truth"] = 66744
# Adjust problem 8 ground truth
test_suite[7]["truth"] = 28921


benchmark_results = []
total_start_time = time.time()

print("="*60)
print("🚀 INITIATING ULTIMATE 20-PROBLEM BENCHMARK SUITE")
print("="*60)

for idx, item in enumerate(test_suite):
    problem_num = idx + 1
    problem_text = item["problem"]
    ground_truth = item["truth"]
    
    print(f"\n[{problem_num}/20] SOLVING...")
    
    start_time = time.time()
    
    try:
        # Trigger the solver
        predicted_answer = solver.solve_problem(problem_text)
    except Exception as e:
        print(f"CRITICAL PIPELINE FAILURE on Problem {problem_num}: {e}")
        predicted_answer = -1 # Indicate failure
    
    time_taken = time.time() - start_time
    
    # Validation logic
    try:
        is_correct = (int(predicted_answer) == int(ground_truth))
    except:
        is_correct = False
    
    benchmark_results.append({
        'Problem ID': problem_num,
        'Question': problem_text,
        'Predicted_Answer': predicted_answer,
        'Ground_Truth': ground_truth,
        'Is_True': is_correct,
        'Execution Time (s)': round(time_taken, 2)
    })
    
    # Aggressive memory cleanup to prevent OOM across 20 heavy loops
    gc.collect()

total_time_taken = time.time() - total_start_time

# ==============================================================================
# DISPLAY FINAL RESULTS
# ==============================================================================
print("\n" + "="*80)
print(f"📊 BENCHMARK COMPLETE IN {round(total_time_taken / 60, 2)} MINUTES")
print("="*80)

df_results = pd.DataFrame(benchmark_results)

# Calculate Accuracy
accuracy = (df_results['Is_True'].sum() / len(df_results)) * 100
print(f"🏆 OVERALL ACCURACY: {accuracy:.1f}% ({df_results['Is_True'].sum()}/20)")
print("="*80 + "\n")'''

In [ ]:
'''import json
import pandas as pd
from IPython.display import display, Markdown

def view_reasoning_logs(log_file='aimo_reasoning_log.jsonl', attempt_filter=None):
    logs = []
    try:
        with open(log_file, 'r') as f:
            for line in f:
                logs.append(json.loads(line.strip()))
    except FileNotFoundError:
        print("Log file not found.")
        return

    df = pd.DataFrame(logs)
    
    display_df = df[['attempt', 'answer', 'python_calls', 'python_errors', 'entropy']].copy()
    display(display_df)

    for log in logs:
        if attempt_filter and log['attempt'] != attempt_filter:
            continue
            
        print("\n" + "="*80)
        print(f"ATTEMPT {log['attempt']} | Final Answer: {log['answer']} | Entropy: {log['entropy']:.3f}")
        print("="*80)
        
        for msg in log['history']:
            role = msg.get('role', '').upper()
            content = msg.get('content', '')
            
            if role == 'ROLE.USER':
                continue
            
            if role == 'ROLE.TOOL':
                print("\n[PYTHON EXECUTION OUTPUT]")
                print("-" * 40)
                print(content.strip())
                print("-" * 40)
            else:
                print(f"\n[MODEL THOUGHTS & CODE]")
                print(content.strip())

view_reasoning_logs()'''